In [1]:
# %load_ext cudf.pandas

In [13]:
import pathlib
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

COMP_DIR = pathlib.Path("../data/")

# COMPETITION DATA
train = pd.read_csv(COMP_DIR / "train.csv", index_col="id")
test = pd.read_csv(COMP_DIR / "test.csv", index_col="id")
sample_submission = pd.read_csv(COMP_DIR / "sample_submission.csv")

train.columns = train.columns.str.lower()
test.columns = test.columns.str.lower()

print(train.shape, test.shape)

(577347, 11) (247435, 10)


In [14]:
train.head(5)

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
id,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [15]:
ohe_df = pd.get_dummies(train[["spectral_type", "galaxy_population"]], drop_first=True, dtype="int")
ohe_df.columns = ohe_df.columns.str.lower()

train = pd.concat([train, ohe_df], axis=1)
train = train.drop(columns=["spectral_type", "galaxy_population"])

In [16]:
TARGET = "class"
NUM_COLS = [c for c in train.select_dtypes(include=["number"]).columns]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + [TARGET]]

In [20]:
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}

train[TARGET] = train[TARGET].map(TARGET_MAPPING)

In [30]:
XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 3,
    "tree_method": "hist",
    "device": "cuda",
    "n_estimators": 10000,
    "learning_rate": 0.01,
    "max_depth": 6,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "random_state": 42,
    "enable_categorical": True
}

In [31]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

X = train[NUM_COLS + CAT_COLS]
y = train[TARGET]

oof_probs = np.zeros((X.shape[0], 3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    model = XGBClassifier(
        **XGB_PARAMS
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        sample_weight=train_weights,
        verbose=0
    )

    oof_probs[val_idx] = model.predict_proba(X_val)
    oof_score = balanced_accuracy_score(y_val, np.argmax(oof_probs[val_idx], axis=1))

    print(f'Fold {fold+1} Balanced Accuracy: {oof_score:.5f}')

oof_probs /= 5

oof_score_full = balanced_accuracy_score(y, np.argmax(oof_probs, axis=1))

print(f"Full OOF balanced_accuracy: {oof_score_full:.5f}")

Fold 1 Balanced Accuracy: 0.96555
Fold 2 Balanced Accuracy: 0.96510
Fold 3 Balanced Accuracy: 0.96505
